In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv("pressure_analysis.csv")
df.head()

Out[2]: 
   match_id  innings  overid  ...  runs_needed  required_rr  pressure
0     13511        1       1  ...        101.0          0.0     False
1     13511        1       1  ...        101.0          0.0     False
2     13511        1       1  ...        101.0          0.0     False
3     13511        1       1  ...        101.0          0.0     False
4     13511        1       1  ...        101.0          0.0     False

[5 rows x 44 columns]


,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,non_striker_name,bowler_id,bowler_name,batting_type,bowling_type,batting_order,runs,is_four,is_six,is_wicket,dismissal_type_id,dismissal_type,dismissed_player_id,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time,first_inn_revised_overs,second_inn_revised_overs,revised_target,score_before,wickets_before,max_overs,balls_remaining,target,runs_needed,required_rr,pressure
0,13511,1,1,1,1,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,0,False,False,False,NaN,NaN,NaN,NaN,6,0,Pace,0.0,Cover Drive,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0,NaN,119.0,101.0,101.0,0.0,False
1,13511,1,1,2,2,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,0,False,False,False,NaN,NaN,NaN,NaN,6,0,Pace,0.0,Flick,Inswing,Over,On-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0,NaN,118.0,101.0,101.0,0.0,False
2,13511,1,1,3,3,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,0,False,False,False,NaN,NaN,NaN,NaN,6,0,Pace,0.0,Square Cut,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0,NaN,117.0,101.0,101.0,0.0,False
3,13511,1,1,4,4,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,0,False,False,False,NaN,NaN,NaN,NaN,6,0,Pace,0.0,Backfoot defensive,Inswing,Over,On-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0,NaN,116.0,101.0,101.0,0.0,False
4,13511,1,1,5,5,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,6,False,True,False,NaN,NaN,NaN,NaN,6,0,Pace,0.0,Pull shot,Seamed In,Over,On-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0,NaN,115.0,101.0,101.0,0.0,False


# Check required columns

In [3]:
required = [
    "bowler_id", "bowler_name", "match_id",
    "runs", "extras_type", "is_wicket",
    "dismissal_type", "pressure"
]

In [4]:
for col in required:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")

# Clean data

In [5]:
df["runs"] = pd.to_numeric(
    df["runs"], errors="coerce"
).fillna(0)

df["extras_type"] = (
    df["extras_type"].fillna("").astype(str).str.lower().str.strip()
)

df["dismissal_type"] = (
    df["dismissal_type"].fillna("").astype(str).str.lower().str.strip()
)

df["is_wicket"] = (
    df["is_wicket"].astype(str).str.lower()
    .isin(["true", "1", "yes"])
)

df["pressure"] = (
    df["pressure"].astype(str).str.lower()
    .isin(["true", "1", "yes"])
)

# Legal balls: exclude wides and no-balls

In [6]:
df["legal_ball"] = (
    ~df["extras_type"].str.contains(
        "wide|no.?ball|overstep", regex=True, na=False
    )
).astype(int)

# Runs charged to bowler

In [7]:
df["bowler_runs"] = df["runs"]

df.loc[
    df["extras_type"].str.contains(
        "bye|leg.?bye|penalty", regex=True, na=False
    ),
    "bowler_runs"
] = 0

# Wickets credited to bowler

In [8]:
not_bowler_wickets = [
    "run out",
    "runout",
    "retired hurt",
    "retired out",
    "retire out",
    "obstructing the field",
    "obstracting the field"
]

df["bowler_wicket"] = (
    df["is_wicket"]
    & ~df["dismissal_type"].isin(not_bowler_wickets)
).astype(int)

# Calculate overall performance

In [9]:
def calculate_stats(data, prefix):
    return data.groupby(
        ["bowler_id", "bowler_name"]
    ).agg(
        runs=("bowler_runs", "sum"),
        balls=("legal_ball", "sum"),
        matches=("match_id", "nunique"),
        wickets=("bowler_wicket", "sum")
    ).reset_index().rename(columns={
        "runs": prefix + "_runs",
        "balls": prefix + "_balls",
        "matches": prefix + "_matches",
        "wickets": prefix + "_wickets"
    })

overall = calculate_stats(df, "overall")

# Calculate pressure performance

In [10]:
pressure = calculate_stats(
    df[df["pressure"]], "pressure"
)

result = pressure.merge(
    overall,
    on=["bowler_id", "bowler_name"],
    how="left"
)

# Calculate economy and wicket rates

In [11]:
result["pressure_economy"] = (
    result["pressure_runs"] * 6 /
    result["pressure_balls"].replace(0, np.nan)
)

result["overall_economy"] = (
    result["overall_runs"] * 6 /
    result["overall_balls"].replace(0, np.nan)
)

result["pressure_wicket_rate"] = (
    result["pressure_wickets"] * 6 /
    result["pressure_balls"].replace(0, np.nan)
)

result["overall_wicket_rate"] = (
    result["overall_wickets"] * 6 /
    result["overall_balls"].replace(0, np.nan)
)

# Compare pressure performance with baseline

In [12]:
result["economy_improvement"] = (
    result["overall_economy"] -
    result["pressure_economy"]
)

result["wicket_rate_uplift"] = (
    result["pressure_wicket_rate"] -
    result["overall_wicket_rate"]
)

# Minimum eligibility

In [13]:
result = result[
    (result["pressure_balls"] >= 60) &
    (result["pressure_matches"] >= 5)
].copy()

# Standard score

In [14]:
def zscore(series):
    sd = series.std(ddof=0)

    if pd.isna(sd) or sd == 0:
        return pd.Series(0.0, index=series.index)

    return (series - series.mean()) / sd

# Weighted bowler score

In [15]:
result["economy_z"] = -zscore(
    result["pressure_economy"]
)

result["wicket_z"] = zscore(
    result["pressure_wicket_rate"]
)

result["economy_improvement_z"] = zscore(
    result["economy_improvement"]
)

result["wicket_uplift_z"] = zscore(
    result["wicket_rate_uplift"]
)

raw_score = (
    0.40 * result["economy_z"] +
    0.25 * result["wicket_z"] +
    0.20 * result["economy_improvement_z"] +
    0.15 * result["wicket_uplift_z"]
)

# Confidence adjustment

In [16]:
confidence = (
    result["pressure_balls"] /
    (result["pressure_balls"] + 60)
) * (
    result["pressure_matches"] /
    (result["pressure_matches"] + 3)
)

result["rating"] = raw_score * confidence

# Rank bowlers

In [17]:
result = result.sort_values(
    ["rating", "pressure_balls"],
    ascending=[False, False]
).reset_index(drop=True)

result.insert(0, "rank", range(1, len(result) + 1))

# Save Top 10

In [18]:
os.makedirs("output", exist_ok=True)

result.head(10).to_csv(
    "output/top_10_bowlers.csv",
    index=False
)
result.head(10).to_csv(
    "top_10_bowlers.csv",
    index=False
)

print("\nTOP 10 BOWLERS")
print(
    result[
        ["rank", "bowler_name", "pressure_wickets",
         "pressure_balls", "pressure_economy",
         "pressure_wicket_rate", "rating"]
    ].head(10).to_string(index=False)
)

print("\nBowler ranking completed!")
print("Saved: top_10_bowlers.csv and output/top_10_bowlers.csv")


TOP 10 BOWLERS
 rank       bowler_name  pressure_wickets  pressure_balls  pressure_economy  pressure_wicket_rate   rating
    1    Mohammad Nawaz                28             286          5.706294              0.587413 1.455674
    2 Matthew Humphreys                13              95          5.873684              0.821053 1.200424
    3    Sufiyan Muqeem                22             246          5.243902              0.536585 1.163222
    4       Abrar Ahmed                26             351          5.726496              0.444444 1.086340
    5      Sunil Narine                37             599          5.989983              0.370618 1.040904
    6  Mitchell Santner                47             653          6.762634              0.431853 0.910668
    7      George Linde                10              96          6.562500              0.625000 0.885114
    8      Hasan Mahmud                15             163          7.214724              0.552147 0.773397
    9     Jamie Overt